
**Mosaic AI Agent Framework**: Author and deploy a tool-calling LangGraph agent
This notebook shows how to author an LangGraph agent and wrap it using the ResponsesAgent interface to make it compatible with Mosaic AI. In this notebook you learn to:

Author a tool-calling LangGraph agent wrapped with ResponsesAgent
Manually test the agent's output
Evaluate the agent using Mosaic AI Agent Evaluation
Log and deploy the agent
To learn more about authoring an agent using Mosaic AI Agent Framework, see Databricks documentation (https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent | Azure).

**Define the agent in code**

Define the agent code in a single cell below. This lets you easily write the agent code to a local Python file, using the %%writefile magic command, for subsequent logging and deployment.

**Agent tools**
This agent code adds the built-in Unity Catalog function system.ai.python_exec to the agent. The agent code also includes commented-out sample code for adding a vector search index to perform unstructured data retrieval.

For more examples of tools to add to your agent, see Databricks documentation (AWS | Azure)

**Wrap the LangGraph agent using the ResponsesAgent interface
For compatibility with Databricks AI features, the LangGraphResponsesAgent class implements the ResponsesAgent interface to wrap the LangGraph agent.

Databricks recommends using ResponsesAgent as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's ResponsesAgent documentation.

In [0]:
%pip install langchain langgraph 

In [0]:
%pip install langgraph --upgrade
dbutils.library.restartPython()

In [0]:
import json
import mlflow
from uuid import uuid4
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict, Union

from databricks_langchain import ChatDatabricks, UCFunctionToolkit, VectorSearchRetrieverTool
from langchain_core.messages import AIMessage, HumanMessage, AIMessageChunk, BaseMessage, convert_to_openai_messages
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda

from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse, ResponsesAgentStreamEvent

# Define LLM endpoint and system prompt

LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = "You are a helpful assistant that cn run Python Code"

# Define tools for your agent, enabling it to retrieve data or take actions
# beyond text generation
# To create and see usage examples of more tools, see
# https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html

tools = []
"""
You can use UDFs in unity Catalog as agent tools. Below, we add the 'system.ai.python_exec' UDf, which provides a python code interpreter tool to our agent. You can also add local langchain python tools. See: https://python.langchain.com/docs/concepts/tools
"""

UC_TOOL_NAMES = ['system.ai.python_exec'] # -> add additional tools as needed here
uc_toolkit = UCFunctionToolkit(function_names = UC_TOOL_NAMES)
tools.extend(uc_toolkit.tools)

# Use Databricks vector search indexes as tools 
# See -> https://docs.databricks.com/en/generative-ai/agent-framework/unstructured-retrieval-tools.html#locally-develop-vector-search-retriever-tools-with-ai-bridge
# List to store vector search tool instance for unstractured retrieval
VECTOR_SEARCH_TOOLS = []
# To add vector search retriever tools. use VectorSearchRetrieverTool and create_tool_info then append the result to tools.
# Eg., 
#VECTOR_SEARCH_TOOLS.apppend(VectorSearchRetrieverTool(
#    index_name, 
#    # filters = "...",
#))
tools.extend(VECTOR_SEARCH_TOOLS)


# Define agent Logic
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

def create_tool_calling_agent(
        model: LanguageModelLike, 
        tools: Union[ToolNode, Sequence[BaseTool]], 
        system_prompt: Optional[str] = None
    ):
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to 
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]

        # If there is a function call, continue, else -> end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return 'continue'
        else:
            return 'end'
    
    if system_prompt:
        preprocessor = RunnableLambda(lambda state: [{
            'role': 'system', 
            'content': system_prompt
        }] + state['messages'])
    else:
        preprocessor = RunnableLambda(lambda state: state['messages'])

    model_runnable = preprocessor | model

    def call_model(state: AgentState, config: RunnableConfig):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}
    
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    def __response_to_cc(self, message: dict[str, Any]) -> list[dict[str, Any]]: 
        """Convert from a Responses API output item to ChatCompletion messages"""
        msg_type = message.get("type")
        if msg_type == "function_call":
            return [{
                "role": "assistant",
                "content": "tool call",
                "tool_calls": [
                    {
                        "id": message["call_id"],
                        "type": "function",
                        "function": {
                            "arguments": message["arguments"],
                            "name": message["name"],
                        },
                    }
                ],
            }]
        elif msg_type == "message":
            content = message["content"]
            if isinstance(content, list):
                return [
                    {
                        "role": message["role"],
                        "content": item["text"]
                    } for item in content
                ]
            else:
                return [{"role": message["role"], "content": content}]
        elif msg_type == "reasoning":
            return [{
                "role": "assistant",
                "content": json.dumps(message["summary"]),
            }]
        
        elif msg_type == "function_call_output":
            return [{
                "role": "tool",
                "content": message["output"],
                "tool_call_id": message["call_id"]
            }]

        else:
            raise ValueError(f"Unknown message type: {msg_type}")
        
        compatible_keys = ["role", "content", "name", "tool_calls", "tool_call_id"]
        filtered = [{k: v for k, v in message.items() if k in compatible_keys}]

        return filtered if filtered else []

    def __prep_msgs_for_cc_llm(self, reponse_input) -> list[dict[str, Any]]:
        """Convert from a Responses API input to ChatCompletion messages"""
        cc_msgs = []
        for msg in response_input:
            cc_msgs.extend(self.__response_to_cc(msg.model_dump()))
        return cc_msgs
    def __langchain_to_responses(self, messages: list[dict[str, Any]]) -> list[ dict[str, Any] ]:
        """Convert from a ChatCompletion messages to Responses API output item dicts"""
        responses = []
        for message in messages:
            message = message.model_dump()
            role = message["type"]
         
            if role == "ai":
                if tool_calls := message.get("tool_calls"):
                    return [
                        self.create_function_call_item(
                            id=message.get("id") or str(uuid4()),
                            call_id = tool_call["id"],
                            name=tool_call["name"],
                            arguments=json.dumps(tool_call["args"])
                        ) for tool_call in tool_calls
                    ]
                else:
                    return [
                        self.create_text_output_item(
                            text=message["content"],
                            id=message.get("id") or str(uuid4())
                        ) 
                    ]       
            elif role == "tool":
                return [
                    self.create_function_call_output_item(
                        call_id=message["tool_call_id"],
                        output=message["content"]
                    )
                ]
            elif role == "user":
                return [message]
    

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item for event in self.predict_stream(request) if event.type == "response.output_item.done"
        ]

        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)
    

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Make a prediction using the agent.""" 
        cc_msgs = []
        for msg in request.input:
            cc_msgs.extend(self.__response_to_cc(msg.model_dump()))
        
        for event in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates", "messages"]):
            if event[0] == "updates":
                for node_data in event[1].values():
                    for item in self.__langchain_to_responses(node_data["messages"]):
                        yield ResponsesAgentStreamEvent(type="response.output_item.done", item=item)

            # filter the streamed messages to just the generated text messages 
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id)
                        )
                except Exception as e:
                    print(e)


# Create the agent object, and specify it as the agent object to use when 
# Loading the agent back for interface via mlflow.model.set_model()

mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphResponsesAgent(agent)
mlflow.models.set_model(AGENT)

In [0]:

result = AGENT.predict({"input": [{"role": "user", "content": "What is 6*7 in Python?"}]})
print(result.model_dump(exclude_none=True))